In [ ]:
import numpy as np
from typing import *

def sparsegpt_simple(W, sparsity):
    # 逐列: 剪掉绝对值最小的行, 把损失均摊到存活行
    Wp = W.copy().astype(np.float64)
    nr, nc = Wp.shape; npr = int(nr*sparsity)
    for c in range(nc):
        w = Wp[:,c].copy(); idx = np.argsort(np.abs(w))
        pi = idx[:npr]; ki = idx[npr:]
        if len(ki)==0: continue
        loss = np.sum(w[pi]); Wp[pi,c]=0.; Wp[ki,c]+=loss/len(ki)
    return Wp.astype(np.float32)

def sparsegpt_hessian(W, X, sparsity):
    # SparseGPT 完整版: 用 Hessian 逆矩阵算最优补偿量
    # 不是简单的均摊, 而是考虑激活分布 X^T X
    import numpy.linalg as la
    Wp = W.copy().astype(np.float64)
    nr, nc = Wp.shape; npr = int(nr*sparsity)
    H = X.T @ X + np.eye(nc)*1e-4; Hi = la.inv(H)
    for c in range(nc):
        w = Wp[:,c].copy(); idx = np.argsort(np.abs(w))
        pi = idx[:npr]; ki = idx[npr:]
        if len(pi)==0: continue
        for j in pi:
            if abs(w[j])<1e-12: continue
            d = -w[j]/(Hi[j,j]+1e-12)*Hi[:,j]
            Wp[ki,c] += d[ki]; Wp[j,c] = 0.
    return Wp.astype(np.float32)

def movement_prune(w0, w1, sparsity):
    # Movement Pruning: 按训练中移动距离 |w1-w0| 剪枝
    # 移动大的权重对 loss 贡献大, 应该保留
    mov = np.abs(w1-w0); thr = np.percentile(mov, sparsity*100)
    return (mov > thr).astype(np.float32)

def global_prune(layers, sparsity):
    # 跨层全局剪枝: 所有层的值放一起排序
    # 值大的层保留更多, 值小的层被剪更多
    all_v = np.concatenate([w.flatten() for w in layers])
    thr = np.percentile(np.abs(all_v), sparsity*100)
    return [w*(np.abs(w)>thr).astype(np.float32) for w in layers]

def iterative_prune(W, target, n_rounds=5):
    # 迭代剪枝: 分多轮逐步提高稀疏度
    # 比一步到位更温和, 每轮只剪当前存活权重的 ~(1-(1-target)^(1/n))
    Wp = W.copy(); per_round = 1 - (1-target)**(1/n_rounds)
    for _ in range(n_rounds):
        nz = Wp != 0; nnz = np.sum(nz)
        if nnz==0: break
        k = int(nnz*per_round)
        if k==0: continue
        thr = np.sort(np.abs(Wp[nz]))[k]
        Wp[(np.abs(Wp)<=thr) & nz] = 0.
    return Wp


In [ ]:
if __name__ == '__main__':
    np.random.seed(42)

    print('1. SparseGPT: magnitude vs simple vs hessian')
    W = np.random.randn(8, 16)*.5; X = np.random.randn(200, 16)
    def mse(Wp): return np.mean((W-Wp)**2)
    _, Wm = prune_by_magnitude(W, .5)
    Ws = sparsegpt_simple(W, .5)
    Wh = sparsegpt_hessian(W, X, .5)
    print(f'  magnitude:  MSE={mse(Wm):.6f}')
    print(f'  SparseGPT:  MSE={mse(Ws):.6f}  ({mse(Wm)/mse(Ws):.1f}x better)')
    print(f'  +Hessian:   MSE={mse(Wh):.6f}  ({mse(Wm)/mse(Wh):.1f}x better)')

    print('\n2. Movement pruning (sparsity=80%)')
    w0 = np.random.randn(20)*.1; w1 = w0.copy()
    w1[:10] += np.random.randn(10)*.5
    mm = movement_prune(w0, w1, .8)
    mmg, _ = prune_by_magnitude(w1.reshape(1,-1), .8)
    print(f'  movement preserves big movers: {int(np.sum(mm[:10]))}/10')
    print(f'  magnitude may cut them:        {int(np.sum(mmg[0,:10]))}/10')

    print('\n3. Global vs per-layer pruning (sparsity=80%)')
    W1 = np.random.randn(10,10)*2.; W2 = np.random.randn(10,10)*.1
    [W1g, W2g] = global_prune([W1, W2], .8)
    s1 = np.sum(W1g==0)/W1g.size*100; s2 = np.sum(W2g==0)/W2g.size*100
    print(f'  W1 (large range): {s1:.0f}% zeros')
    print(f'  W2 (small range): {s2:.0f}% zeros (pruned more)')

    print('\n4. Iterative vs one-shot (target=95%)')
    W = np.random.randn(20,20)*.5
    Wi = iterative_prune(W, .95, 5)
    _, Wo = prune_by_magnitude(W, .95)
    ei = np.mean((W-Wi)**2); eo = np.mean((W-Wo)**2)
    print(f'  one-shot:  MSE={eo:.6f}')
    print(f'  iterative: MSE={ei:.6f}  ({eo/ei:.1f}x better)')

    print('\n5. Lottery Ticket Hypothesis')
    N, D = 200, 10
    X = np.random.randn(N, D); y = X @ (np.random.randn(D)*2) + np.random.randn(N)*.1
    w_init = np.linalg.solve(X.T@X, X.T@y)
    w_t = w_init + np.random.randn(D)*.1
    m = prune_by_magnitude(w_t.reshape(1,-1), .5)[0].flatten()
    w_lt = w_init*m + np.random.randn(D)*.1*m
    print(f'  init MSE:     {np.mean((y-X@w_init)**2):.4f}')
    print(f'  trained MSE:  {np.mean((y-X@w_t)**2):.4f}')
    print(f'  lottery MSE:  {np.mean((y-X@w_lt)**2):.4f}  (reset + retrain)')

def prune_by_magnitude(W, sparsity):
    thr = np.percentile(np.abs(W), sparsity*100)
    return (np.abs(W)>thr).astype(np.float32), W
